In [16]:
from pyprojroot import here
import polars as pl
from sklearn.model_selection import train_test_split
import cv2

In [6]:
# params
path_raw = here() / "data" / "raw"
path_img = path_raw / "Images"
test_size = 0.2
seed = 123

In [7]:
df = pl.read_csv(path_raw / "Labels.csv").drop("")
df.head()

Image Name,Patient,Label,Quality Score
str,i64,str,f64
"""0_0.jpg""",0,"""GON+""",6.18
"""1_0.jpg""",1,"""GON+""",5.31
"""1_1.jpg""",1,"""GON+""",4.37
"""2_0.jpg""",2,"""GON+""",3.51
"""2_1.jpg""",2,"""GON+""",4.24


# Train test split
> avoid data leakage because same patient can have different image

In [8]:
patient_unique_id = df.select(pl.col("Patient").unique()).to_numpy().flatten()
train_id, test_id = train_test_split(
    patient_unique_id, test_size=test_size, random_state=seed
)
df = df.with_columns(
    pl.when(pl.col("Patient").is_in(test_id))
    .then(pl.lit("TEST"))
    .otherwise(pl.lit("TRAIN"))
    .alias("SPLIT")
)

df.head()

Image Name,Patient,Label,Quality Score,SPLIT
str,i64,str,f64,str
"""0_0.jpg""",0,"""GON+""",6.18,"""TRAIN"""
"""1_0.jpg""",1,"""GON+""",5.31,"""TRAIN"""
"""1_1.jpg""",1,"""GON+""",4.37,"""TRAIN"""
"""2_0.jpg""",2,"""GON+""",3.51,"""TRAIN"""
"""2_1.jpg""",2,"""GON+""",4.24,"""TRAIN"""


# Image resolution

In [22]:
img_name_ls = df.select("Image Name").to_numpy().flatten().tolist()
res_ls = [cv2.imread(path_img / i).shape for i in img_name_ls]
height_ls = [i[0] for i in res_ls]
width_ls = [i[1] for i in res_ls]

# add new col
df = df.with_columns(
    pl.Series("Height", height_ls),
    pl.Series("Width", width_ls),
)

df.head()

Image Name,Patient,Label,Quality Score,SPLIT,Height,Width
str,i64,str,f64,str,i64,i64
"""0_0.jpg""",0,"""GON+""",6.18,"""TRAIN""",1918,1918
"""1_0.jpg""",1,"""GON+""",5.31,"""TRAIN""",1929,1929
"""1_1.jpg""",1,"""GON+""",4.37,"""TRAIN""",1940,1940
"""2_0.jpg""",2,"""GON+""",3.51,"""TRAIN""",1893,1893
"""2_1.jpg""",2,"""GON+""",4.24,"""TRAIN""",1900,1900
